## 1. Evaluation initiale sur le dataset Adult

### 1.1 Chargement et exécution des modèles sur le dataset Adult
Elle charge les données Adult et exécute les trois modèles (XGBoost, BRCG, RIPPER) pour comparer leurs performances.

In [ ]:
import import_ipynb
import brcg_algorithm
import adultdata_preparation_evaluat
import xgboost_rule_extraction
import explainer
import pandas as pd

from ripper_algorithm import train_ripper, evaluate_ripper

# Charger les données
x_train, x_test, y_train, y_test, mapping = adultdata_preparation_evaluat.prepare_adult_dataset()
print("Données chargées depuis le notebook principal")

# Nettoyer les données
x_train = x_train.drop(columns=["label"], errors="ignore")
x_test = x_test.drop(columns=["label"], errors="ignore")

# Exécuter le notebook XGBoost
get_ipython().run_line_magic("run", "./xgboost_rule_extraction.ipynb")

print("train_xgboost disponible:", "train_xgboost" in dir())

if "train_xgboost" in dir():
    x_test = x_test.drop(columns=["label"], errors="ignore")
    xgb_model, y_pred_xgb, xgb_metrics = xgboost_rule_extraction.train_xgboost(
        x_train, y_train, x_test, y_test
    )
    print("Métriques XGBoost:", xgb_metrics)
else:
    print("ERREUR: train_xgboost n'a pas été chargé correctement")
    print("Contenu de l'espace de noms:", [v for v in dir() if "train" in v])

# BRCG
brcg_model_, fb_brcg, columns_brcg, train_time_brcg, brcg_metrics, brcg_rules = brcg_algorithm.train_brcg(
    x_train, y_train, x_test, y_test
)

brcg_eval = brcg_algorithm.evaluate_brcg(
    model=brcg_model_,
    fb=fb_brcg,
    X_train=x_train,
    y_train=y_train,
    X_test=x_test,
    y_test=y_test,
    elapsed=None,
    ram=None,
    columns=columns_brcg,
    rules=brcg_rules
)

print("Métriques BRCG:", brcg_eval)

# RIPPER
ripper_model_, ripper_metrics, ripper_rules = train_ripper(
    x_train, y_train, x_test, y_test
)

ripper_eval = evaluate_ripper(
    model=ripper_model_,
    X_train=x_train,
    y_train=y_train,
    X_test=x_test,
    y_test=y_test,
    elapsed=None,
    ram=None
)

print("Métriques RIPPER:", ripper_eval)

Données chargées depuis le notebook principal


100%|██████████| 14/14 [00:00<00:00, 14.55it/s]


Total Support: 23.48% (6116)
After removing duplicates:
Unique Coverage: 21.88% (5698)


100%|██████████| 723/723 [00:00<00:00, 3223.43it/s]


Test set coverage: 18.26%
Test coverage F1-score: 0.98
--------------------------------------
Total Skipped: 5324 (81.74%)
 - Due to Conflicts: 0 (0.0%)
 - Due to No Rule Coverage: 5324 (81.74%)
--------------------------------------
Using Majority class to predict non-covered cases.
F1-score (if using entire dataset): 0.7775


100%|██████████| 723/723 [00:00<00:00, 2890.35it/s]


Test set coverage: 18.26%
Test coverage F1-score: 0.98
--------------------------------------
Total Skipped: 5324 (81.74%)
 - Due to Conflicts: 0 (0.0%)
 - Due to No Rule Coverage: 5324 (81.74%)
--------------------------------------
Using model to predict non-covered cases.
F1-score (if using entire dataset): 0.8689
train_xgboost disponible: True
Métriques XGBoost: {'accuracy': 0.8753262705358513, 'balanced_accuracy': 0.7969251078262138, 'precision': 0.7990543735224587, 'recall': 0.645448758752387, 'f1': 0.7140845070422536, 'accuracy_train': 0.8748080466830467, 'f1_train': 0.7087092451987494, 'training_time': 0.47576093673706055}
Learning DNF rule with complexity parameters lambda0=0.001, lambda1=0.001
Initial LP solved
Iteration: 1, Objective: 0.2181
Iteration: 2, Objective: 0.1897
Iteration: 3, Objective: 0.1843
Iteration: 4, Objective: 0.1843
Iteration: 5, Objective: 0.1843
Métriques BRCG: {'F1_train': 0.8043357582201471, 'F1_test': 0.8026137894465102, 'Train_Coverage': nan, 'Train

### 1.2 Comparaison des performances des modèles

In [4]:
print("\n===== Comparaison des modèles =====")
print(f"XGBoost  -> Accuracy: {xgb_metrics['accuracy']:.4f} | F1-score: {xgb_metrics['f1']:.4f}")
print(f"BRCG     -> Accuracy: {brcg_eval['Test_Exactitude']:.4f} | F1-score: {brcg_eval['F1_test']:.4f}")
print(f"RIPPER   -> Accuracy: {ripper_eval['Test_Exactitude']:.4f} | F1-score: {ripper_eval['F1_test']:.4f}")


===== Comparaison des modèles =====
XGBoost  -> Accuracy: 0.8753 | F1-score: 0.7141
BRCG     -> Accuracy: 0.8190 | F1-score: 0.8026
RIPPER   -> Accuracy: 0.8176 | F1-score: 0.7887


--> XGBoost surpasse en accuracy mais souffre d'un déséquilibre de classes (F1-score plus faible)

--> RIPPER offre le meilleur compromis avec un F1-score élevé et une bonne accuracy

--> BRCG est très proche de RIPPER en performance

## 2. Évaluation multicritères sur les 5 datasets (BRCG RIPPER)

### 2.1 Import des bibliothèques pour l'évaluation 

In [ ]:
import traceback
import brcg_algorithm
from brcg_algorithm import train_brcg, evaluate_brcg
from ripper_algorithm import train_ripper, evaluate_ripper
from cmr_pipeline import train_cmr, evaluate_cmr, predict_cmr_test_only
from cmr_pure import train_cmr_pure, evaluate_cmr_pure, predict_cmr_pure_test_only
import joblib
import explainer
import io
import os
import contextlib
import re
import import_ipynb
import ripper_algorithm
import numpy as np
from tqdm import tqdm
import psutil
import time
import threading
import gc
import matplotlib.pyplot as plt
import pandas as pd 

In [32]:
# Mesure le temps d'exécution et le pic de mémoire RAM 
def run_with_peak_ram(func, *args, interval=0.05, **kwargs):
    process = psutil.Process()
    baseline_rss = process.memory_info().rss
    peak_rss = {"value": baseline_rss}
    stop_flag = {"stop": False}
    result_holder = {}

    # Fonction de monitoring de la mémoire
    def monitor():
        while not stop_flag["stop"]:
            try:
                rss = process.memory_info().rss
                if rss > peak_rss["value"]:
                    peak_rss["value"] = rss
            except:
                pass
            time.sleep(interval)

    t = threading.Thread(target=monitor, daemon=True)

    start = time.time()
    t.start()

    try:
        result_holder["result"] = func(*args, **kwargs)
    finally:
        stop_flag["stop"] = True
        t.join()

    elapsed = time.time() - start

    added_ram_mb = max(0.0, (peak_rss["value"] - baseline_rss) / 1024**2)
    peak_ram_mb = peak_rss["value"] / 1024**2

    return result_holder["result"], elapsed, added_ram_mb, peak_ram_mb

In [ ]:
n_iterations = 1
np.random.seed(42)

os.makedirs("results", exist_ok=True)
os.makedirs("results/distributions", exist_ok=True)

# Chargement datasets
DATASETS = {
    "KDD99": "../Datasets/KDD99/",
    "BIG15": "../Datasets/BIG15/",
    "UNSW-NB15": "../Datasets/UNSW-NB15/processed/",
    "DoH20": "../Datasets/DoH20/"
}

# Définition des colonnes pour les résultats
COLUMNS = [
    "Dataset", "Méthode", "F1_train", "F1_test", "Train_Coverage", "Train_Exactitude",
    "Test_Coverage", "Test_Exactitude", "Conflit", "Non_couverts", "F1_rmaj", "F1_rr",
    "Nb_règles", "Nb_conditions", "Complexité", "Temps", "RAM_MB", "Iteration"
]

# Nettoyage des données et des labels, en remplaçant les infinis par des NaN, puis en remplissant les NaN par 0 et en convertissant les types
def clean_data(df):
    df = df.copy()
    df = df.replace([np.inf, -np.inf], np.nan)
    df = df.apply(pd.to_numeric, errors="coerce")
    df = df.fillna(0)
    return df.astype(np.float32)

def clean_labels(y):
    y = pd.Series(y)
    y = y.replace([np.inf, -np.inf], np.nan)
    y = pd.to_numeric(y, errors="coerce").fillna(0)
    return y.astype(int).values

#Alignement spécial pour BRCG : remplace les valeurs de test qui n'apparaissent pas dans l'entraînement par 0
def align_test_for_brcg(X_train, X_test):
    X_train_brcg = X_train.copy()
    X_test_brcg = X_test.copy()

    for col in X_train_brcg.columns:
        train_values = set(X_train_brcg[col].unique())
        X_test_brcg[col] = X_test_brcg[col].apply(lambda x: x if x in train_values else 0)

    return X_train_brcg, X_test_brcg

# Fonction pour charger les datasets, avec un traitement spécial pour UNSW-NB15 
def load_dataset(name, path):
    if name == "UNSW-NB15":
        train = pd.read_csv(path + "le_train.csv")
        test = pd.read_csv(path + "le_test.csv")

        X_train = train.drop(["id", "attack_cat", "label"], axis=1)
        y_train = train["label"]

        X_test = test.drop(["id", "attack_cat", "label"], axis=1)
        y_test = test["label"]
    else:
        X_train = pd.read_csv(path + "X_train.csv")
        X_test = pd.read_csv(path + "X_test.csv")
        y_train = pd.read_csv(path + "y_train.csv").squeeze()
        y_test = pd.read_csv(path + "y_test.csv").squeeze()

    return X_train, y_train, X_test, y_test

# Préparation finale avant les modèles
def prepare_data(X_train, y_train, X_test, y_test):
    X_train = clean_data(X_train)
    X_test = clean_data(X_test)

    y_train = clean_labels(y_train)
    y_test = clean_labels(y_test)

    X_train = X_train.loc[:, X_train.nunique() > 1]
    X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

    return X_train, y_train, X_test, y_test

# Mesure le temps d'exécution et le pic de mémoire RAM pour une fonction donnée, en utilisant un thread de monitoring pour suivre la mémoire pendant l'exécution de la fonction. 
# Retourne le résultat de la fonction, le temps écoulé, la RAM ajoutée en MB et le pic de RAM en MB. Utilise psutil pour obtenir les informations de mémoire du processus et time pour mesurer le temps d'exécution. 
# Le monitoring de la mémoire s'arrête une fois quela fonction a terminé son exécution, et le thread de monitoring est joint pour s'assurer qu'il se termine proprement. 
# La RAM ajoutée est calculée en soustrayant la mémoire de baseline du pic de mémoire.
def run_with_peak_ram(func, *args, interval=0.05, **kwargs):
    process = psutil.Process()
    baseline_rss = process.memory_info().rss
    peak_rss = {"value": baseline_rss}
    stop_flag = {"stop": False}
    result_holder = {}

    def monitor():
        while not stop_flag["stop"]:
            try:
                rss = process.memory_info().rss
                if rss > peak_rss["value"]:
                    peak_rss["value"] = rss
            except:
                pass
            time.sleep(interval)

    t = threading.Thread(target=monitor, daemon=True)

    start = time.time()
    t.start()

    try:
        result_holder["result"] = func(*args, **kwargs)
    finally:
        stop_flag["stop"] = True
        t.join()

    elapsed = time.time() - start
    added_ram_mb = max(0.0, (peak_rss["value"] - baseline_rss) / 1024**2)

    return result_holder["result"], elapsed, added_ram_mb


# |------------------------- BRCG -------------------------|

def evaluate_brcg_block(X_train, y_train, X_test, y_test):
    X_train_brcg, X_test_brcg = align_test_for_brcg(X_train, X_test)

    model, fb, columns, _, _, rules = train_brcg(
        X_train_brcg, y_train, X_test_brcg, y_test
    )

    def test_only():
        X_test_bin = fb.transform(X_test_brcg)
        return model.predict(X_test_bin)

    _, elapsed, ram = run_with_peak_ram(test_only)

    return evaluate_brcg(
        model, fb, X_train_brcg, y_train, X_test_brcg, y_test,
        elapsed=elapsed, ram=ram, columns=columns, rules=rules
    )

# |------------------------- RIPPER -------------------------|

def evaluate_ripper_block(X_train, y_train, X_test, y_test):
    model, _, _ = train_ripper(X_train, y_train, X_test, y_test)

    def test_only():
        test_data = X_test.copy()
        test_data["label"] = y_test
        return model.predict(test_data)

    _, elapsed, ram = run_with_peak_ram(test_only)

    return evaluate_ripper(
        model, X_train, y_train, X_test, y_test,
        elapsed=elapsed, ram=ram
    )

# |------------------------- CMR -------------------------|

def evaluate_cmr_block(X_train, y_train, X_test, y_test):
    payload = train_cmr(X_train, y_train, X_test, y_test)

    _, elapsed, ram = run_with_peak_ram(predict_cmr_test_only, payload)

    return evaluate_cmr(payload, elapsed=elapsed, ram=ram)

# |----------------------- CMR PURE -------------------------|

def evaluate_cmr_pure_block(X_train, y_train, X_test, y_test):
    payload = train_cmr_pure(X_train, y_train, X_test, y_test)

    _, elapsed, ram = run_with_peak_ram(predict_cmr_pure_test_only, payload)

    return evaluate_cmr_pure(payload, elapsed=elapsed, ram=ram)

# Fonction pour exécuter les modèles sur un même dataset et retourner leurs métriques respectives dans un dictionnaire
def test_models(X_train, y_train, X_test, y_test):
    X_train, y_train, X_test, y_test = prepare_data(X_train, y_train, X_test, y_test)

    return {
        "BRCG": evaluate_brcg_block(X_train, y_train, X_test, y_test),
        "RIPPER": evaluate_ripper_block(X_train, y_train, X_test, y_test),
        "CMR": evaluate_cmr_block(X_train, y_train, X_test, y_test),
        "CMR_pur": evaluate_cmr_pure_block(X_train, y_train, X_test, y_test)
    }



# Boucle principale pour exécuter les modèles sur tous les datasets et toutes les itérations, en stockant les résultats dans un dictionnaire et en sauvegardant les progrès dans un fichier CSV à chaque itération.
#  Utilise tqdm pour afficher une barre de progression pendant l'exécution. Après toutes les itérations, calcule la moyenne des métriques pour chaque méthode et chaque dataset, 
# et sauvegarde le tableau comparatif final dans un fichier CSV. Génère également des graphiques de distribution pour chaque métrique si le fichier de progrès existe.
results_all_iterations = {ds: [] for ds in DATASETS}
progress_rows = []

for name, path in tqdm(DATASETS.items()):
    X_train, y_train, X_test, y_test = load_dataset(name, path)

    for it in range(n_iterations):
        res = test_models(X_train, y_train, X_test, y_test)
        results_all_iterations[name].append(res)

        for method in ["BRCG", "RIPPER", "CMR", "CMR_pur"]:
            row = [name, method] + [res[method].get(col, np.nan) for col in COLUMNS[2:-1]] + [it + 1]
            progress_rows.append(row)

        pd.DataFrame(progress_rows, columns=COLUMNS).to_csv(
            "results/progress_iterations.csv",
            index=False,
            encoding="utf-8-sig",
            na_rep="NaN"
        )

    gc.collect()

# Calcul du tableau final moyen pour chaque méthode et chaque dataset, et sauvegarde le tableau comparatif final dans un fichier CSV.
final_rows = []

for ds in DATASETS.keys():
    runs = results_all_iterations.get(ds, [])

    for method in ["BRCG", "RIPPER", "CMR", "CMR_pur"]:
        valid_runs = [r[method] for r in runs if r is not None and r.get(method) is not None]

        if valid_runs:
            df_metrics = pd.DataFrame(valid_runs)
            avg = df_metrics.mean(numeric_only=True).to_dict()
            row = [ds, method] + [avg.get(c, None) for c in COLUMNS[2:-1]]
            final_rows.append(row)

df_results = pd.DataFrame(final_rows, columns=COLUMNS[:-1])
df_results = df_results.sort_values(by=["Dataset", "Méthode"])

df_results.to_csv(
    "results/tableau_comparatif.csv",
    index=False,
    encoding="utf-8-sig",
    na_rep="NaN"
)

# Génère des graphiques de distribution pour chaque métrique si le fichier de progrès existe.
if os.path.exists("results/progress_iterations.csv"):
    df_progress = pd.read_csv("results/progress_iterations.csv")

    for metric in ["F1_test", "Temps", "RAM_MB"]:
        if metric in df_progress.columns and not df_progress[metric].isna().all():
            plt.figure()
            df_progress.boxplot(column=metric, by=["Dataset", "Méthode"])
            plt.title(metric)
            plt.suptitle("")
            plt.xticks(rotation=45)
            plt.tight_layout()
            plt.savefig(f"results/distributions/{metric}.png")
            plt.close()
# Affiche le tableau comparatif final dans la console
print(df_results)

  0%|          | 0/4 [00:00<?, ?it/s]

Learning DNF rule with complexity parameters lambda0=0.001, lambda1=0.001
Initial LP solved


 25%|██▌       | 1/4 [07:32<22:38, 452.73s/it]

Learning DNF rule with complexity parameters lambda0=0.001, lambda1=0.001
Initial LP solved


 50%|█████     | 2/4 [20:12<21:06, 633.10s/it]

...

100%|██████████| 4/4 [1:58:51<00:00, 1782.85s/it]


      Dataset Méthode  F1_train   F1_test  Train_Coverage  Train_Exactitude  \
3       BIG15    BRCG  0.981789  0.975395             NaN          0.981736   
5       BIG15     CMR  0.994849  0.984834           95.15          1.000000   
4       BIG15  RIPPER  0.983891  0.983512             NaN          0.983864   
9       DoH20    BRCG  0.960502  0.461520             NaN          0.964770   
11      DoH20     CMR  1.000000  0.999981          100.00          1.000000   
10      DoH20  RIPPER  0.946630  0.946485             NaN          0.938440   
0       KDD99    BRCG  0.973266  0.973377             NaN          0.973340   
2       KDD99     CMR  0.999554  0.997733           69.74          1.000000   
1       KDD99  RIPPER  0.998729  0.998351             NaN          0.998729   
6   UNSW-NB15    BRCG  0.816047  0.767133             NaN          0.816050   
8   UNSW-NB15     CMR  0.974531  0.918022           66.59          1.000000   
7   UNSW-NB15  RIPPER  0.915584  0.732781           

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

### 2.2 Évaluation avec 1 iterations

#### Avec 1 iteration pour le datasets CIC2017

### 2.3 Évaluation avec 10 iterations avec KDD99, BIG15, UNSW-NB15, DoH20

100%|██████████| 4/4 [1:20:16<00:00, 1204.11s/it]


      Dataset Méthode  F1_train   F1_test  Train_Coverage  Train_Exactitude  \
3       BIG15    BRCG  0.981789  0.975395             NaN          0.981736   
4       BIG15  RIPPER  0.984681  0.984533             NaN          0.984652   
5       BIG15     CMR  0.999584  0.986700           95.41          1.000000   
9       DoH20    BRCG  0.960502  0.461520             NaN          0.964770   
10      DoH20  RIPPER  0.951262  0.952135             NaN          0.944535   
11      DoH20     CMR  1.000000  1.000000          100.00          1.000000   
0       KDD99    BRCG  0.972841  0.973002             NaN          0.972911   
1       KDD99  RIPPER  0.998377  0.997939             NaN          0.998377   
2       KDD99     CMR  0.999991  0.998100           70.05          1.000000   
6   UNSW-NB15    BRCG  0.816047  0.767133             NaN          0.816050   
7   UNSW-NB15  RIPPER  0.910285  0.825874             NaN          0.910156   
8   UNSW-NB15     CMR  0.989292  0.917200           

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

### 2.3 Évaluation avec 10 iterations avec CIC2017 sampling 25K

In [ ]:
n_iterations = 10
np.random.seed(42)

os.makedirs("results", exist_ok=True)
os.makedirs("results/distributions", exist_ok=True)

def clean_data(df):
    df = df.copy()
    df = df.replace([np.inf, -np.inf], np.nan)
    df = df.apply(pd.to_numeric, errors="coerce")
    df = df.fillna(0)
    return df.astype(np.float32)

def clean_labels(y):
    y = pd.Series(y)
    y = y.replace([np.inf, -np.inf], np.nan)
    y = pd.to_numeric(y, errors="coerce").fillna(0)
    return y.astype(int).values

def test_models_fast(X_train, y_train, X_test, y_test, sample_size=None, dataset_name="", iteration=0):
    results = {}
    process = psutil.Process()

    X_train = clean_data(X_train)
    X_test = clean_data(X_test)

    y_train = clean_labels(y_train)
    y_test = clean_labels(y_test)

    if sample_size is not None and len(X_train) > sample_size:
        rng = np.random.default_rng(42 + iteration)
        idx = rng.choice(len(X_train), sample_size, replace=False) # pour avoir des sample differrent a chaque iteration 
        X_train_sample = X_train.iloc[idx].copy()
        y_train_sample = y_train[idx].copy()
    else:
        X_train_sample = X_train.copy()
        y_train_sample = y_train.copy()

    X_train_sample = X_train_sample.loc[:, X_train_sample.nunique() > 1]
    X_test_aligned = X_test.reindex(columns=X_train_sample.columns, fill_value=0)

    X_train_sample = X_train_sample.astype(np.float32)
    X_test_aligned = X_test_aligned.astype(np.float32)

    y_train_sample = y_train_sample.astype(int).ravel()
    y_test_arr = y_test.astype(int).ravel()

         # -------- BRCG --------
    X_train_brcg = X_train_sample.copy()
    X_test_brcg = X_test_aligned.copy()

    for col in X_train_brcg.columns:
        train_values = set(X_train_brcg[col].unique())
        X_test_brcg[col] = X_test_brcg[col].apply(lambda x: x if x in train_values else 0)

    try:
        (output_brcg, elapsed, added_ram, peak_ram_total) = run_with_peak_ram(
            train_brcg,
            X_train_brcg,
            pd.Series(y_train_sample),
            X_test_brcg,
            pd.Series(y_test_arr)
        )

        model_brcg, fb_brcg, columns_brcg, train_time_brcg, metrics_brcg, rules_brcg = output_brcg

        results["BRCG"] = evaluate_brcg(
            model_brcg,
            fb_brcg,
            X_train_brcg,
            y_train_sample,
            X_test_brcg,
            y_test_arr,
            elapsed,
            added_ram,
            columns=columns_brcg
        )

    except Exception as e:
        print(f"Erreur BRCG: {e}")
        results["BRCG"] = None
        
        # -------- RIPPER --------
    try:
        (output_ripper, elapsed, added_ram, peak_ram_total) = run_with_peak_ram(
            train_ripper,
            X_train_sample,
            pd.Series(y_train_sample),
            X_test_aligned,
            pd.Series(y_test_arr)
        )

        model_ripper, y_pred_ripper, metrics, rules = output_ripper

        results["RIPPER"] = evaluate_ripper(
            model_ripper,
            X_train_sample,
            y_train_sample,
            X_test_aligned,
            y_test_arr,
            elapsed,
            added_ram
        )

    except Exception as e:
        print(f"Erreur RIPPER: {e}")
        results["RIPPER"] = None
   
    # -------- CMR --------
    try:
        (output_cmr, elapsed, added_ram, peak_ram_total) = run_with_peak_ram(
            run_cmr,
            X_train_sample,
            y_train_sample,
            X_test_aligned,
            y_test_arr
        )

        results["CMR"] = output_cmr
        results["CMR"]["Temps"] = float(elapsed)
        results["CMR"]["RAM_MB"] = float(added_ram)

    except Exception as e:
        print(f"Erreur CMR: {e}")
        results["CMR"] = None


    return results


datasets = {
    "CIC2017": "../Datasets/CIC2017/"
}

results_all_iterations = {"CIC2017_25k_10iterations": []}
progress_rows = []

columns = [
    "Dataset","Méthode","F1_train","F1_test","Train_Coverage","Train_Exactitude",
    "Test_Coverage","Test_Exactitude","Conflit","Non_couverts","F1_rmaj","F1_rr",
    "Complexité","Temps","RAM_MB","Iteration"
]

for name, path in tqdm(datasets.items()):
    try:
        X_train = pd.read_parquet(path + "X_train.parquet")
        X_test = pd.read_parquet(path + "X_test.parquet")
        y_train = pd.read_csv(path + "y_train.csv").squeeze()
        y_test = pd.read_csv(path + "y_test.csv").squeeze()

        for it in range(n_iterations):
            res = test_models_fast(
                X_train,
                y_train,
                X_test,
                y_test,
                sample_size=25000,
                dataset_name="CIC2017_25k_10iterations",
                iteration=it+1
            )
            results_all_iterations["CIC2017_25k_10iterations"].append(res)

            for method in ["BRCG", "RIPPER", "CMR"]:
                if res.get(method) is not None:
                    row = ["CIC2017_25k_10iterations", method] + list(res[method].values()) + [it+1]
                    progress_rows.append(row)

            df_progress = pd.DataFrame(progress_rows, columns=columns)
            df_progress.to_csv("results/progress_iterations_cic2017_25k_10iterations.csv", index=False)

        del X_train, X_test, y_train, y_test
        gc.collect()

    except Exception as e:
        print(f"\nErreur globale sur le dataset {name}: {e}")
        traceback.print_exc()
        results_all_iterations[name] = [None] * n_iterations
        gc.collect()

final_rows = []

for ds, runs in results_all_iterations.items():

    for method in ["BRCG", "RIPPER", "CMR"]:
        valid_runs = [r[method] for r in runs if r is not None and r.get(method) is not None]
        if valid_runs:
            df_metrics = pd.DataFrame(valid_runs)
            avg = df_metrics.mean(numeric_only=True).to_dict()
            row = ["CIC2017_25k_10iterations", method] + [avg.get(c, None) for c in columns[2:-1]]
            final_rows.append(row)

df_results = pd.DataFrame(final_rows, columns=columns[:-1])

df_results = df_results.sort_values(
    by=["Dataset", "Méthode"],
    key=lambda col: col.map({"BRCG": 0, "RIPPER": 1, "CMR": 2}) if col.name == "Méthode" else col
)

df_results.to_csv("results/tableau_comparatif_cic2017_25k_10iterations.csv", index=False)

if os.path.exists("results/progress_iterations_cic2017_25k_10iterations.csv"):
    df_progress = pd.read_csv("results/progress_iterations_cic2017_25k_10iterations.csv")

    metrics_to_plot = ["F1_test", "Temps", "RAM_MB"]

    for metric in metrics_to_plot:
        plt.figure()
        df_progress.boxplot(column=metric, by=["Dataset", "Méthode"])
        plt.title(metric)
        plt.suptitle("")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.savefig(f"results/distributions/{metric}_cic2017_25k_10iterations.png")
        plt.close()

print(df_results)

  0%|          | 0/1 [00:00<?, ?it/s]

Learning DNF rule with complexity parameters lambda0=0.001, lambda1=0.001
Initial LP solved
Iteration: 1, Objective: 0.0786
Iteration: 2, Objective: 0.0619
Iteration: 3, Objective: 0.0619


100%|██████████| 68/68 [01:49<00:00,  1.61s/it]


Learning DNF rule with complexity parameters lambda0=0.001, lambda1=0.001
Initial LP solved
Iteration: 1, Objective: 0.0762
Iteration: 2, Objective: 0.0609


100%|██████████| 68/68 [03:15<00:00,  2.88s/it]


...

                    Dataset Méthode  F1_train   F1_test  Train_Coverage  \
0  CIC2017_25k_10iterations    BRCG  0.943136  0.941329             NaN   
1  CIC2017_25k_10iterations  RIPPER  0.997675  0.996095             NaN   
2  CIC2017_25k_10iterations     CMR  0.998708  0.997620          92.783   

   Train_Exactitude  Test_Coverage  Test_Exactitude  Conflit  Non_couverts  \
0          0.946952            NaN         0.945348      NaN           NaN   
1          0.997676            NaN         0.996096      NaN           NaN   
2          1.000000         92.263         0.993254    2.488         7.737   

    F1_rmaj     F1_rr  Complexité       Temps      RAM_MB  
0       NaN  0.941329         1.0  643.620000   545.99500  
1       NaN  0.996095        30.7   85.229000   625.82000  
2  0.938002  0.992452     15791.3  369.752181  6441.56875  


<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

: 

### utils

In [ ]:
import platform
import psutil
import sys
import sklearn
import numpy
import pandas
import pandas as pd
import os

print("Python:", sys.version)
print("sklearn:", sklearn.__version__)
print("numpy:", numpy.__version__)
print("pandas:", pandas.__version__)

print("OS :", platform.system(), platform.release())
print("CPU :", platform.processor())
print("RAM :", round(psutil.virtual_memory().total / (1024**3), 2), "GB")

datasets = {
    "KDD99": "../Datasets/KDD99/",
    "BIG15": "../Datasets/BIG15/",
    "DoH20": "../Datasets/DoH20/",
    "UNSW-NB15": "../Datasets/UNSW-NB15/processed/",
    "CIC2017": "../Datasets/CIC2017/"
}

results = []

for name, path in datasets.items():
    try:
        if name == "CIC2017":
            X_train = pd.read_parquet(path + "X_train.parquet")
            X_test = pd.read_parquet(path + "X_test.parquet")

        elif name == "UNSW-NB15":
            train = pd.read_csv(path + "le_train.csv")
            test = pd.read_csv(path + "le_test.csv")

            X_train = train.drop(["id", "attack_cat", "label"], axis=1)
            X_test = test.drop(["id", "attack_cat", "label"], axis=1)

        else:
            X_train = pd.read_csv(path + "X_train.csv")
            X_test = pd.read_csv(path + "X_test.csv")

        results.append({
            "Dataset": name,
            "Train Samples": X_train.shape[0],
            "Test Samples": X_test.shape[0],
            "Features": X_train.shape[1]
        })

    except Exception as e:
        print(f"Erreur avec {name}: {e}")

df_sizes = pd.DataFrame(results)
print(df_sizes)

OS : Windows 10
CPU : Intel64 Family 6 Model 78 Stepping 3, GenuineIntel
RAM : 15.41 GB


Les expériences ont été réalisées dans l’environnement technique suivant :

Système d’exploitation : Windows 10
Processeur : Intel64 Family 6 Model 78 (GenuineIntel)
Mémoire RAM : 15.41 Go
Langage de programmation : Python 3.9.25 (64 bits)

Bibliothèques utilisées :

scikit-learn : 1.6.1
NumPy : 1.23.5
pandas : 1.5.3

Les expériences ont été exécutées en local sur cette machine.


Resume tableau 8 : Les résultats montrent qu'augmenter le paramètre de support minimal (s_min) réduit le nombre de règles extraites, ce qui diminue les taux de conflit mais aussi la couverture. 

Si les performances peuvent s'améliorer légèrement grâce à la réduction du bruit dans certains jeux de données, elles peuvent également se dégrader lorsque des motifs importants sont supprimés. Notamment, s_min = 1 et s_min = 3 produisent des résultats identiques, indiquant que toutes les règles extraites bénéficient déjà d'un support suffisant. Globalement, s_min = 3 offre un bon compromis en termes de couverture.…